# 05 — ResNet18 Transfer Learning + Grad-CAM

**Subject:** Advanced ML / Deep Learning (Alejandro E.)  
**Environment:** Google Colab — T4 GPU required  
**Goal:** Fine-tune ResNet18 on travel Twitter images using transfer learning.

## Pipeline
1. Mount Google Drive and load the dataset CSV
2. Fix image paths to point to Drive
3. Build image datasets with data augmentation
4. Load ResNet18 pretrained on ImageNet — freeze all layers except `fc`
5. Train for 10 epochs on T4 GPU
6. Evaluate on test set and save probabilities for late fusion
7. Generate Grad-CAM visualisations

**Expected F1:** ~0.40–0.45  
**Expected training time:** ~15 minutes on T4 GPU

## Cell 1 — Mount Drive and load dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/multimodal_emotion/data/labels.csv')
print(f"Dataset loaded: {df.shape}")
print(df['label'].value_counts())

## Cell 2 — Imports and GPU check

⚠️ Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
import json, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
# Expected output: Device: cuda
# If it shows 'cpu', enable GPU in Runtime settings.

## Cell 3 — Fix image paths for Colab

The CSV was created on Mac with local paths (`data/raw/data/X.jpg`).
We redirect them to the Drive folder where the images were uploaded.

In [ ]:
DRIVE_IMG_DIR = '/content/drive/MyDrive/multimodal_emotion/data/raw/data'

def fix_path(local_path):
    """Redirect local Mac path to Google Drive path."""
    filename = os.path.basename(local_path)
    return os.path.join(DRIVE_IMG_DIR, filename)

df['image_path'] = df['image_path'].apply(fix_path)

# Verify images exist
existing = df['image_path'].apply(os.path.exists)
print(f"Images found: {existing.sum()} / {len(df)}")
# Expected: Images found: 4869 / 4869

df = df[existing].reset_index(drop=True)
print(f"Final dataset: {df.shape}")

## Cell 4 — Image transforms and Dataset class

**Training transforms** include data augmentation to reduce overfitting:
- `RandomHorizontalFlip` — doubles effective dataset size
- `ColorJitter` — makes model robust to lighting variations

**Validation/Test transforms** use only resize + normalise (no augmentation).

**Normalisation values** are the ImageNet mean and std — required because
ResNet18 was pretrained on ImageNet with these exact statistics.

In [ ]:
TRAIN_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),           # data augmentation
    transforms.ColorJitter(0.2, 0.2, 0.1),       # brightness/contrast jitter
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],  # ImageNet mean
                         [0.229, 0.224, 0.225])  # ImageNet std
])

VAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class ImageDataset(Dataset):
    """PyTorch Dataset for loading image-label pairs."""
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), self.labels[idx]

## Cell 5 — Data splits

We use three splits:
- **Train (70%)**: used to update model weights
- **Validation (10%)**: used to monitor training and avoid overfitting
- **Test (20%)**: held out — never seen during training

Same `random_state=42` as all other modules for fair comparison.

In [ ]:
le     = LabelEncoder()
labels = le.fit_transform(df['label'].tolist())
paths  = df['image_path'].tolist()
n_cls  = len(le.classes_)
print(f"Classes: {le.classes_}")

# 80/20 test split
tr_p, te_p, tr_l, te_l = train_test_split(
    paths, labels, test_size=0.2, random_state=42, stratify=labels)

# Further split train into train/val (87.5% / 12.5% of original train)
tr_p, va_p, tr_l, va_l = train_test_split(
    tr_p, tr_l, test_size=0.125, random_state=42)

tr_dl = DataLoader(ImageDataset(tr_p, tr_l, TRAIN_TF),
                   batch_size=32, shuffle=True, num_workers=2)
va_dl = DataLoader(ImageDataset(va_p, va_l, VAL_TF),
                   batch_size=32, num_workers=2)
te_dl = DataLoader(ImageDataset(te_p, te_l, VAL_TF),
                   batch_size=32, num_workers=2)

print(f"Train: {len(tr_p)} | Val: {len(va_p)} | Test: {len(te_p)}")

## Cell 6 — Build ResNet18 with Transfer Learning

**Transfer learning strategy:**
- Load ResNet18 pretrained on ImageNet (1.2M images, 1000 classes)
- **Freeze all layers** — keep the learned features from ImageNet
- **Replace the final `fc` layer** — adapt to our 3-class problem
- Only train the new `fc` layer (much faster, avoids overfitting)

This is appropriate because ImageNet features (edges, textures, shapes)
are useful for emotion recognition in travel images.

In [ ]:
def build_resnet(num_classes):
    """Build ResNet18 with frozen backbone and new classification head."""
    model = models.resnet18(weights='IMAGENET1K_V1')

    # Freeze all layers — keep ImageNet features
    for p in model.parameters():
        p.requires_grad = False

    # Replace final layer for our 3-class task
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_resnet(n_cls).to(device)
crit  = nn.CrossEntropyLoss()

# Only train the fc layer parameters
opt   = torch.optim.Adam(model.fc.parameters(), lr=2e-4)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=4, gamma=0.5)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print("Model ready.")

## Cell 7 — Training (10 epochs)

⏱️ **Expected time: ~15 minutes on T4 GPU**  
The loss should decrease from ~1.10 to ~1.04.

In [ ]:
train_losses = []

for epoch in range(10):
    model.train()
    total = 0
    for imgs, lbls in tr_dl:
        opt.zero_grad()
        loss = crit(model(imgs.to(device)), lbls.to(device))
        loss.backward()
        opt.step()
        total += loss.item()
    sched.step()
    avg = total / len(tr_dl)
    train_losses.append(avg)
    print(f"Epoch {epoch+1}/10 — Loss: {avg:.4f}")

## Cell 8 — Evaluation on test set

In [ ]:
model.eval()
all_preds, all_probas = [], []

with torch.no_grad():
    for imgs, _ in te_dl:
        probs = torch.softmax(model(imgs.to(device)), dim=1).cpu().numpy()
        all_probas.extend(probs.tolist())
        all_preds.extend(np.argmax(probs, axis=1).tolist())

print("=== ResNet18 Transfer Learning ===")
print(classification_report(te_l, all_preds, target_names=le.classes_))
print(f"F1 macro: {f1_score(te_l, all_preds, average='macro'):.4f}")

## Cell 9 — Save results and model to Drive

In [ ]:
os.makedirs('/content/drive/MyDrive/multimodal_emotion/results', exist_ok=True)

# Save metrics and probabilities for late fusion
results = {
    'f1_macro':      f1_score(te_l, all_preds, average='macro'),
    'probas':        all_probas,
    'train_losses':  train_losses,
    'label_encoder': le.classes_.tolist()
}
with open('/content/drive/MyDrive/multimodal_emotion/results/metrics_resnet.json', 'w') as f:
    json.dump(results, f, indent=2)

# Save trained model weights
torch.save({
    'model_state_dict': model.state_dict(),
    'label_encoder':    le.classes_.tolist(),
    'num_classes':      n_cls
}, '/content/drive/MyDrive/multimodal_emotion/results/resnet18_model.pth')

print("Results saved to Drive.")
print("Remember to download metrics_resnet.json and resnet18_model.pth to your local repo.")

## Cell 10 — Training loss curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(range(1, 11), train_losses, marker='o', color='#ea580c', linewidth=2)
plt.title('Training Loss — ResNet18 Transfer Learning')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks(range(1, 11))
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/multimodal_emotion/results/resnet_loss_curve.png', dpi=150)
plt.show()
print("Loss curve saved to Drive.")

## Cell 11 — Grad-CAM visualisation

**Grad-CAM** (Gradient-weighted Class Activation Mapping) shows which regions
of the image the model focused on when making a prediction.

It works by computing the gradients of the predicted class score with respect
to the feature maps of the last convolutional layer (`layer4`). High gradient
values indicate regions that strongly influenced the prediction.

Red = high attention, Blue = low attention.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image

def generate_gradcam(model, img_tensor, target_class, device):
    """Generate Grad-CAM heatmap for a given image and target class."""
    model.eval()
    img_tensor  = img_tensor.unsqueeze(0).to(device)
    gradients   = []
    activations = []

    def save_gradient(grad):
        gradients.append(grad)

    def forward_hook(module, input, output):
        activations.append(output)
        output.register_hook(save_gradient)

    hook   = model.layer4.register_forward_hook(forward_hook)
    output = model(img_tensor)
    model.zero_grad()
    output[0, target_class].backward()
    hook.remove()

    grad    = gradients[0].squeeze().cpu().numpy()
    act     = activations[0].squeeze().cpu().detach().numpy()
    weights = grad.mean(axis=(1, 2))

    cam = np.zeros(act.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * act[i]

    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (224, 224))
    cam -= cam.min()
    cam /= cam.max() + 1e-7
    return cam

def tensor_to_img(tensor):
    """Convert normalised tensor back to displayable image."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.cpu().numpy().transpose(1, 2, 0)
    img  = std * img + mean
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)

In [ ]:
# Find one confident example per class for Grad-CAM
model.eval()
examples    = {'negative': None, 'neutral': None, 'positive': None}
class_names = le.classes_
global_idx  = 0

with torch.no_grad():
    for imgs, lbls in te_dl:
        probs = torch.softmax(model(imgs.to(device)), dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

        for i, (pred, lbl) in enumerate(zip(preds, lbls.numpy())):
            class_name = class_names[lbl]
            confidence = probs[i][pred]
            if pred == lbl and confidence > 0.70 and examples[class_name] is None:
                examples[class_name] = {
                    'batch_imgs': imgs,
                    'idx':        i,
                    'label':      class_name,
                    'confidence': confidence
                }
        global_idx += len(imgs)
        if all(v is not None for v in examples.values()):
            break

for cls, ex in examples.items():
    if ex:
        print(f"{cls}: confidence {ex['confidence']:.2f}")

In [ ]:
# Generate Grad-CAM figure for all three classes
class_idx    = {name: i for i, name in enumerate(le.classes_)}
class_colors = {'negative': '#dc2626', 'neutral': '#6b7280', 'positive': '#16a34a'}

fig, axes = plt.subplots(3, 3, figsize=(12, 11))

for row, (cls_name, ex) in enumerate(examples.items()):
    if ex is None:
        continue
    img_tensor   = ex['batch_imgs'][ex['idx']]
    target_class = class_idx[cls_name]

    cam     = generate_gradcam(model, img_tensor, target_class, device)
    orig    = tensor_to_img(img_tensor)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.6 * cv2.resize(orig, (224, 224)) + 0.4 * heatmap).astype(np.uint8)

    axes[row, 0].imshow(cv2.resize(orig, (224, 224)))
    axes[row, 0].set_title('Original', fontsize=11)
    axes[row, 1].imshow(heatmap)
    axes[row, 1].set_title('Grad-CAM', fontsize=11)
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f'Overlay', fontsize=11)

    color = class_colors[cls_name]
    for ax in axes[row]:
        ax.axis('off')
    axes[row, 0].set_ylabel(cls_name.upper(), fontsize=13,
                             fontweight='bold', color=color)
    axes[row, 0].axis('off')

plt.suptitle('Grad-CAM — ResNet18 Transfer Learning',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

save_path = '/content/drive/MyDrive/multimodal_emotion/results/gradcam_examples.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Grad-CAM saved to Drive.")
print("Remember to download gradcam_examples.png to paper/figures/ in your local repo.")